# Expected Returns Analytics Module (v3.1)

Automated pipeline for expected returns analysis using the v3.1+ analytics platform:
- **Monte Carlo Simulation** — Probabilistic upside/downside distributions
- **Price Target Achievement** — Probability-weighted expected returns by sector
- **Kalman Filtered Targets** — Noise-reduced price target signals
- **Earnings Beat Analysis** — Three-layer Bayesian earnings beat probability
- **Accounting Anomaly Detection** — Multi-metric anomaly scoring (Beneish, Benford, Mahalanobis)
- **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
- **Quad-Model Agreement** — MC + Kalman + Achievement + Earnings Beat
- **Statistical Analysis** — Bayesian category analysis, copula dependency, MCMC
- **Probability Analytics** — Category-level probability distributions, credit risk
- **Stock Screening** — Quality, value, growth, dividend, GARP, health filters

Data sources (v3.2 — Equities MV + Feature Views):
- `public.mv_equities` (equities data via `load_equities_data_from_db`)
- `public.vw_features_*` (17 feature views via `load_all_feature_views`)
- `equities_schema_metadata` (dynamic column discovery via `get_equities_schema`)

In [1]:
##%% Imports & Setup
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats as sp_stats

# --- Data utilities ---
from finance_ml.analytics.data_utils import (
    ExportConfig, aggregate_probability_results, backfill_feature_columns,
    compute_metric_statistics, export_to_csv, export_to_db, export_to_json,
    get_equities_schema, get_identifier_cols_set, get_view_category_mapping,
    load_all_feature_views,
    load_feature_data_from_db, load_equities_data_from_db,
    load_feature_categories_from_db, load_identifier_columns,
    reorder_with_identifiers, validate_feature_alignment,
)

# --- Optimised operations ---
from finance_ml.analytics.optimized_ops import (
    fast_ruin_probability, get_optimization_status,
    vectorized_percentile_rank, vectorized_zscore,
)

# --- Probability models ---
from finance_ml.analytics.probability_analytics import (
    CategoryProbabilityAnalyzer, CreditRiskProbabilityModel,
    DividendCutProbabilityModel, EarningsBeatProbabilityModel,
    EPSStreakAnalyzer, PriceTargetAchievementModel,
    ResampledBeatProbabilityModel, create_earnings_probability_dashboard,
    export_probability_analytics_results,
)

# --- Screening ---
from finance_ml.analytics.screening import (
    create_enhanced_screener, create_sector_relative_ranking,
    rank_stocks_by_composite_score, screen_dividend_quality,
    screen_earnings_quality, screen_financial_health,
    screen_garp_opportunities, screen_growth_momentum,
    screen_high_yield_safe_dividends, screen_integrity_filtered_growth,
    screen_valuation_reversion_candidates, screen_value_opportunities,
)

# --- Statistical analysis ---
from finance_ml.analytics.statistical_analysis import (
    bayesian_category_analysis, bayesian_earnings_beat_model,
    calculate_conditional_probabilities, calculate_ruin_probability,
    detect_accounting_anomalies, fit_distributions_by_category,
    fit_gaussian_copula, hierarchical_mcmc_by_sector,
    kalman_filter_price_target, kalman_momentum_filter,
    mcmc_student_t, monte_carlo_price_target_simulation,
    parallel_mcmc_chains, resampled_posterior_returns,
    run_category_probability_analytics,
    analyze_employee_productivity_frontier, analyze_reporting_lag_sentiment,
)

# --- InferenceData schema (ArviZ) ---
try:
    from finance_ml.analytics.inference_schema import (
        ARVIZ_AVAILABLE, EquityCoordinates,
        build_beat_probability_inference_data, build_credit_risk_inference_data,
        build_monte_carlo_inference_data, summarize_inference_data,
    )
except ImportError:
    ARVIZ_AVAILABLE = False
    EquityCoordinates = None

# --- Visualizations: Probabilistic (ArviZ-backed) ---
from finance_ml.analytics.visualizations._shared import PLOTLY_TEMPLATE
from finance_ml.analytics.visualizations.probability_viz import (
    create_bayesian_category_ridge, create_beat_probability_posterior,
    create_posterior_return_forest, create_ruin_probability_diagnostic,
    create_tri_model_posterior_comparison,
)

# --- Visualizations: Quality & Risk ---
from finance_ml.analytics.visualizations.quality_risk import (
    create_altman_zscore_distribution, create_beneish_mscore_analysis,
    create_distress_early_warning_dashboard, create_piotroski_fscore_breakdown,
    create_quality_risk_quadrant, create_risk_tier_sunburst,
)

# --- Visualizations: Earnings Quality ---
from finance_ml.analytics.visualizations.earnings_quality import (
    create_enhanced_beat_probability_dashboard as create_enhanced_beat_prob_dash,
    create_gaap_divergence_plot, create_revision_momentum_chart,
)

# --- Visualizations: Expected Returns ---
from finance_ml.analytics.visualizations.expected_returns_viz import (
    create_beat_vs_achievement_scatter, create_kalman_vs_raw_scatter,
    create_mc_return_distribution, create_model_dispersion_dashboard,
    create_return_distribution_fit_chart, create_screening_summary_chart,
    create_sector_heatmap, create_sector_return_analytics_heatmap,
    create_sector_risk_reward_scatter, create_strong_consensus_bar,
    create_tri_model_agreement_histogram, create_var_analysis,
)

from finance_ml.logging_config import configure_logging
from finance_ml.ml_workflow.core.utils import safe_divide

px.defaults.template = PLOTLY_TEMPLATE
warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

In [2]:
##%% Pipeline Configuration
@dataclass
class PipelineConfig:
    """Centralized configuration for the expected returns analytics pipeline."""
    mc_simulations: int = 50_000
    mc_max_stocks: int = 10_000
    mcmc_chains: int = 4
    mcmc_samples: int = 10_000
    beat_threshold: float = 0.6
    output_dir: str = "outputs/analytics"
    log_file: str | None = "logs/expected_returns_pipeline.log"
    log_level: int = logging.INFO

    @classmethod
    def from_env(cls) -> "PipelineConfig":
        return cls(
            mc_simulations=int(os.environ.get("ER_MC_SIMULATIONS", 50_000)),
            mc_max_stocks=int(os.environ.get("ER_MC_MAX_STOCKS", 10_000)),
            mcmc_chains=int(os.environ.get("ER_MCMC_CHAINS", 4)),
            mcmc_samples=int(os.environ.get("ER_MCMC_SAMPLES", 10_000)),
            output_dir=os.environ.get("ER_OUTPUT_DIR", "outputs/analytics"),
            log_file=os.environ.get("ER_LOG_FILE", "logs/expected_returns_pipeline.log"),
        )

cfg = PipelineConfig.from_env()
output_dir = Path(cfg.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

configure_logging(level=cfg.log_level, log_file=cfg.log_file, console=True)

In [3]:
##%% Pipeline Helpers
from expected_returns_v3 import (
    _log_and_print, _has_required_columns, get_feature_categories,
    _LazyFeatureCategories, reconcile_feature_categories,
    FEATURE_CATEGORIES,
)

In [4]:
##%% Schema Column Discovery & Backfill
from expected_returns_v3 import _get_schema_columns, _apply_backfill_and_kalman

In [5]:
##%% Model Statistics
from expected_returns_v3 import compute_model_detailed_statistics, print_model_statistics

## 1. Data Loading — Equities & Feature Views (v3.2)

Load data from materialized views:
- `mv_equities` → primary equities data
- `vw_features_*` → 17 feature views
- `mv_all_stock_features` → combined feature MV

Mirrors the SQL queries from `expected_returns_analytics.ipynb`:
- `SELECT * FROM monte_carlo_simulation`
- `SELECT * FROM price_target_achievement`
- `SELECT * FROM kalman_filtered_price_targets`
- `SELECT * FROM vw_features_analyst_sentiment`
- `SELECT * FROM vw_features_cashflow`

In [6]:
##%% Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

In [7]:
##%% Data Loading Functions
from expected_returns_v3 import (
    load_expected_returns_data, load_all_stock_features, load_analytics_table,
)

In [8]:
##%% Execute: Load Data
df = load_expected_returns_data()
print(f"✓ Loaded mv_equities: {len(df):,} stocks × {len(df.columns)} features")

df_all = load_all_stock_features()
if not df_all.empty:
    print(f"✓ Loaded feature views: {len(df_all):,} stocks × {len(df_all.columns)} features")
else:
    df_all = df.copy()

df_features = load_analytics_table()
if not df_features.empty:
    print(f"✓ Loaded mv_all_stock_features: {len(df_features):,} stocks × {len(df_features.columns)} features")

df.head()

2026-03-03 14:18:34,359 - root - INFO - Loading equities data from localhost:5432/postgres (view: public.mv_equities)
2026-03-03 14:18:37,702 - root - INFO - Loaded 6389 rows from public.mv_equities
2026-03-03 14:18:37,705 - root - INFO - Backfill complete. Columns: 674
2026-03-03 14:18:37,745 - root - INFO - Loaded 24 feature categories from database
2026-03-03 14:18:37,746 - expected_returns_v3 - INFO - Loaded 24 feature categories (226 total features)
2026-03-03 14:18:37,747 - expected_returns_v3 - INFO - All feature categories have ≥80%% coverage
2026-03-03 14:18:37,747 - expected_returns_v3 - INFO - Loaded expected returns data: 6389 stocks × 674 features


AttributeError: 'tuple' object has no attribute 'columns'

## 1b. Pre-compute Historical Target Drift Enrichment

Compute historical target drift enrichment once, used by Steps 2, 3, and 4.
Eliminates redundant 3× recomputation.

In [ ]:
##%% Historical Drift Enrichment Functions
from expected_returns_v3 import (
    ALL_HISTORICAL_PRICE_TARGET_COLS, _resolve_available_historical_cols,
    _log_historical_coverage, _enrich_with_historical_target_drift,
)

In [ ]:
##%% Execute: Pre-compute Historical Drift Enrichment
hist_available = _resolve_available_historical_cols(df)
_log_historical_coverage(hist_available)
df_enriched = _enrich_with_historical_target_drift(df.copy(), hist_available)
print(f"✓ Historical drift enrichment complete ({len(df_enriched.columns) - len(df.columns)} derived columns)")

## 2. Monte Carlo Simulation

Probabilistic price target simulation using pre-enriched data (historical drift already applied).

In [ ]:
##%% Monte Carlo Model Functions
from expected_returns_v3 import (
    run_monte_carlo_analysis,
    compute_price_target_mc,
)

In [ ]:
##%% Execute: Monte Carlo Simulation
mc = run_monte_carlo_analysis(df_enriched, n_simulations=cfg.mc_simulations, max_stocks=cfg.mc_max_stocks, use_historical_targets=False)
mc = compute_price_target_mc(mc, df)
print(f"✓ {len(mc):,} stocks simulated")
if not mc.empty:
    print(f"  Mean upside: {mc['expected_upside_pct'].mean():.1f}%")
mc.head()

In [ ]:
##%% Visualization: MC Return Distribution
if not mc.empty:
    fig = create_mc_return_distribution(mc)
    fig.show()

In [ ]:
##%% Visualization: Sector Risk-Reward Scatter
if not mc.empty:
    fig = create_sector_risk_reward_scatter(mc)
    fig.show()

In [ ]:
##%% Visualization: VaR Analysis
if not mc.empty:
    fig = create_var_analysis(mc)
    fig.show()

In [ ]:
##%% Visualization: Posterior Return Forest
if not mc.empty:
    fig = create_posterior_return_forest(mc, top_n=25)
    fig.show()

In [ ]:
##%% Visualization: Return Distribution Fit
if not mc.empty:
    fig = create_return_distribution_fit_chart(mc)
    fig.show()

## 3. Price Target Achievement

Probability-weighted expected returns by sector with analyst conviction scoring.
Uses pre-enriched data (historical drift already applied).

In [ ]:
##%% Price Target Achievement Functions
from expected_returns_v3 import run_price_target_achievement, compute_price_target_prob_weighted

In [ ]:
##%% Execute: Price Target Achievement
pt = run_price_target_achievement(df_enriched, use_historical_targets=False, feature_df=df_all)
pt = compute_price_target_prob_weighted(pt, df)
print(f"✓ {len(pt):,} stocks analyzed")
pt.head()

## 4. Kalman Filtered Targets

Noise-reduced price target signals using Kalman filtering.
Uses pre-enriched data (historical drift already applied).

In [ ]:
##%% Kalman Filter Functions
from expected_returns_v3 import run_kalman_filter

In [ ]:
##%% Execute: Kalman Filter
kal = run_kalman_filter(df_enriched, use_historical_targets=False)
if not kal.empty:
    print(f"✓ {len(kal):,} stocks filtered, mean filtered upside: {kal['filtered_upside'].mean():.1f}%")
else:
    print("⚠️ Kalman filter returned empty results")
kal.head()

In [ ]:
##%% Visualization: Kalman vs Raw Scatter
if not kal.empty:
    fig = create_kalman_vs_raw_scatter(kal)
    fig.show()

## 5. Earnings Beat Analysis

Three-layer Bayesian earnings beat probability model.

In [ ]:
##%% Earnings Beat Functions
from expected_returns_v3 import run_earnings_beat_analysis

In [ ]:
##%% Execute: Earnings Beat
beat = run_earnings_beat_analysis(df_all if not df_all.empty else df)
if not beat.empty and "posterior_beat_prob" in beat.columns:
    print(f"✓ {len(beat):,} stocks, mean P(beat): {beat['posterior_beat_prob'].mean():.3f}")
else:
    print(f"✓ {len(beat):,} stocks processed")
beat.head()

In [ ]:
##%% Visualization: Beat Probability Posterior
if not beat.empty:
    fig = create_beat_probability_posterior(beat, top_n=12)
    fig.show()

In [ ]:
##%% Visualization: Earnings Probability Dashboard
if not beat.empty:
    fig = create_earnings_probability_dashboard(beat)
    fig.show()

In [ ]:
##%% Visualization: Enhanced Beat Probability Dashboard
if not beat.empty:
    fig = create_enhanced_beat_prob_dash(beat)
    fig.show()

In [ ]:
##%% Visualization: EPS Revision Momentum
if not beat.empty and "eps_revision_momentum" in beat.columns:
    fig = create_revision_momentum_chart(beat, top_n=30)
    fig.show()

In [ ]:
##%% Visualization: GAAP Divergence
if not beat.empty and "gaap_adj_eps_gap_pct" in beat.columns:
    fig = create_gaap_divergence_plot(beat)
    fig.show()

## 5b. Accounting Anomaly Detection

Multi-metric anomaly scoring: Beneish M-Score, Benford's Law, Mahalanobis distance,
sector-relative z-scores, and per-feature anomaly flags.
Runs before credit risk so anomaly columns can be merged into the credit DataFrame.

In [ ]:
##%% Accounting Anomaly Functions
from expected_returns_v3 import run_accounting_anomaly_analysis

In [ ]:
##%% Execute: Accounting Anomaly Detection
anomaly_results = run_accounting_anomaly_analysis(df, feature_df=df_all)
if not anomaly_results.empty and "accounting_anomaly_score" in anomaly_results.columns:
    print(f"✓ Accounting anomaly analysis: {len(anomaly_results):,} stocks")
    if "accounting_anomaly_tier" in anomaly_results.columns:
        tier_counts = anomaly_results["accounting_anomaly_tier"].value_counts()
        for tier_label in ["Clean", "Watch", "Flag", "Alert"]:
            count = tier_counts.get(tier_label, 0)
            pct = count / len(anomaly_results) * 100 if len(anomaly_results) > 0 else 0
            print(f"  {tier_label}: {count:,} ({pct:.1f}%)")
else:
    anomaly_results = pd.DataFrame()
    print("⚠️ No accounting anomaly features available")

## 5c. Credit Risk & Dividend Safety

Credit risk and dividend safety analysis. Anomaly columns from Step 5b are merged
into the credit risk DataFrame for downstream alignment.

In [ ]:
##%% Credit Risk & Dividend Safety Functions
from expected_returns_v3 import run_credit_risk_analysis, run_dividend_safety_analysis

In [ ]:
##%% Execute: Credit Risk & Dividend Safety
credit = run_credit_risk_analysis(df, feature_df=df_all)
div_safety = run_dividend_safety_analysis(df_all)

# Merge anomaly columns into credit for downstream alignment
if not credit.empty and not anomaly_results.empty and "ticker" in anomaly_results.columns:
    anom_cols = [c for c in anomaly_results.columns if c != "ticker" and c not in credit.columns]
    if anom_cols:
        credit = credit.merge(anomaly_results[["ticker"] + anom_cols], on="ticker", how="left")
        print(f"  Merged {len(anom_cols)} anomaly columns into credit risk DataFrame")

if not credit.empty:
    print(f"✓ Credit risk: {len(credit):,} stocks")
if not div_safety.empty:
    print(f"✓ Dividend safety: {len(div_safety):,} stocks")

In [ ]:
##%% Visualization: Ruin Probability Diagnostic
if not credit.empty and "ruin_probability" in credit.columns:
    fig = create_ruin_probability_diagnostic(credit, top_n=20)
    fig.show()

## 5d. Stock Screening

Quality, value, growth, dividend, GARP, health, and integrity-filtered screens.

In [ ]:
##%% Stock Screening Functions
from expected_returns_v3 import run_stock_screening, filter_quality_stocks

In [ ]:
##%% Execute: Stock Screening
screens = run_stock_screening(df_all)
for name, screen_df in screens.items():
    if not screen_df.empty:
        print(f"  ✓ {name}: {len(screen_df):,} stocks")

In [ ]:
##%% Visualization: Screening Summary
if screens:
    fig = create_screening_summary_chart(screens)
    fig.show()

## 5e. Resampled Bayesian Posterior Returns (v3.1)

In [ ]:
##%% Execute: Resampled Posterior
from expected_returns_v3 import run_resampled_posterior_analysis

resampled_posterior = run_resampled_posterior_analysis(df)
if not resampled_posterior.empty:
    print(f"✓ {len(resampled_posterior):,} stocks, mean posterior: {resampled_posterior['posterior_mean'].mean()*100:.2f}%")

## 6. Cross-Model Alignment

Tri-model (MC + Kalman + Achievement) and quad-model (+ Earnings Beat) alignment.

In [ ]:
##%% Alignment Functions
from expected_returns_v3 import (
    _SIGNAL_LABELS, _SIGNAL_LABELS_4,
    build_tri_model_alignment, build_quad_model_alignment,
    extract_strong_consensus, compute_cross_model_correlation,
)

In [ ]:
##%% Execute: Cross-Model Alignment
tri = build_tri_model_alignment(mc, kal, pt)
strong = extract_strong_consensus(tri)
quad = build_quad_model_alignment(tri, beat, beat_threshold=cfg.beat_threshold)
corr_info = compute_cross_model_correlation(mc, kal)
if not tri.empty:
    print(f"Tri-model: {len(tri):,}, Strong consensus: {len(strong)}, Quad full: {(quad['quad_agreement']==4).sum() if not quad.empty else 0}")

In [ ]:
##%% Visualization: Tri-Model Agreement
if not tri.empty:
    fig = create_tri_model_agreement_histogram(tri)
    fig.show()

In [ ]:
##%% Visualization: Sector Heatmap
if not tri.empty:
    fig = create_sector_heatmap(tri)
    fig.show()

In [ ]:
##%% Visualization: Strong Consensus Picks
if not strong.empty:
    fig = create_strong_consensus_bar(strong)
    fig.show()

In [ ]:
##%% Visualization: Tri-Model Posterior Comparison
if not strong.empty:
    fig = create_tri_model_posterior_comparison(strong, top_n=12)
    fig.show()

In [ ]:
##%% Visualization: Beat vs Achievement Scatter
if not beat.empty and not pt.empty:
    fig = create_beat_vs_achievement_scatter(beat, pt)
    fig.show()

## 7. Expected Returns Summary (4-Model Merge)

Merges MC, Kalman, Price Target, and Earnings Beat into a unified summary with
quality filtering, z-score ranking, sector analytics, and hierarchical MCMC.
Now includes anomaly results for accounting quality integration.

In [ ]:
##%% Summary & Analytics Functions
from expected_returns_v3 import (
    build_expected_returns_summary,
    compute_derived_price_target, compute_price_target_prob_weighted,
    compute_price_target_mc, compute_sector_expected_returns,
    compute_sector_return_analytics, compute_return_zscore_ranks,
    compute_cross_model_diagnostics, compute_return_distribution_analytics,
    run_parallel_mcmc_return_analysis,
)

In [ ]:
##%% Execute: Build Summary
summary = build_expected_returns_summary(mc, kal, pt, beat, anomaly_results, source_df=df_all)
summary = filter_quality_stocks(summary, df_all)
summary = compute_return_zscore_ranks(summary)
sector_analytics = compute_sector_return_analytics(summary)
if not summary.empty:
    print(f"✓ {len(summary):,} stocks in summary")
summary.head()

In [ ]:
##%% Visualization: Model Dispersion Dashboard
if not summary.empty:
    fig = create_model_dispersion_dashboard(summary)
    fig.show()

In [ ]:
##%% Visualization: Expected Returns Summary Posterior
if not summary.empty:
    fig = create_tri_model_posterior_comparison(summary, top_n=12)
    fig.show()

In [ ]:
##%% Visualization: Sector Return Analytics Heatmap
if not sector_analytics.empty:
    fig = create_sector_return_analytics_heatmap(sector_analytics)
    fig.show()

## 7a. Parallel MCMC Return Analysis (v3.1)

Gelman-Rubin convergence diagnostics across parallel chains.

In [ ]:
##%% Execute: Parallel MCMC
mcmc_result = run_parallel_mcmc_return_analysis(mc, n_chains=cfg.mcmc_chains, n_samples=cfg.mcmc_samples)
if mcmc_result:
    print(f"R̂={mcmc_result.get('r_hat', float('nan')):.4f}, converged={mcmc_result.get('converged', False)}")

## 7b. Per-Category Bayesian Probability Analytics

Runs analysis on all available feature categories from the 17 `vw_features_*` database
views for full feature coverage, using `get_view_category_mapping()` to dynamically
resolve categories and their feature columns.

In [ ]:
##%% Execute: Category Probability Analytics
from expected_returns_v3 import run_category_probability_analysis

# Build categories from all 17 vw_features_* views for full feature coverage
view_mapping = get_view_category_mapping()
all_categories: dict[str, list[str]] = {}
for view_name, info in view_mapping.items():
    cat_label = info.get("category", view_name)
    feat_cols = info.get("feature_cols", [])
    if feat_cols:
        if cat_label in all_categories:
            all_categories[cat_label].extend(
                c for c in feat_cols if c not in all_categories[cat_label]
            )
        else:
            all_categories[cat_label] = list(feat_cols)

print(f"Feature categories from {len(view_mapping)} views → {len(all_categories)} categories, "
      f"{sum(len(v) for v in all_categories.values())} total features")

category_analytics = run_category_probability_analysis(df_all, categories=all_categories)
for cat_name, cat_result in category_analytics.items():
    print(f"  {cat_name}: {cat_result.get('features_analyzed', 0)} features")

In [ ]:
##%% Visualization: Bayesian Analyst Sentiment Ridge
sentiment_features = [f for f in ["analyst_bullish_pct", "upside_potential", "eps_revision_momentum",
    "analyst_conviction", "pt_consensus_convergence"] if f in df_all.columns]
if sentiment_features:
    results = bayesian_category_analysis(df_all, "Analyst Sentiment", sentiment_features)
    fig = create_bayesian_category_ridge(results, category_name="Analyst Sentiment")
    fig.show()

In [ ]:
##%% Visualization: Bayesian Profitability Ridge
prof_features = [f for f in ["roe", "roa", "roic", "gross_margin_pct", "operating_margin_pct"] if f in df_all.columns]
if prof_features:
    results = bayesian_category_analysis(df_all, "Profitability", prof_features)
    fig = create_bayesian_category_ridge(results, category_name="Profitability")
    fig.show()

## 8. InferenceData (ArviZ)

Build ArviZ InferenceData objects for MC, Beat Probability, and Credit Risk models.
Built before visualizations so that charts can consume properly structured InferenceData
with coordinates, log-likelihoods, and convergence diagnostics.

In [ ]:
##%% Build InferenceData
idata_mc = None
idata_beat = None
idata_credit = None
if ARVIZ_AVAILABLE:
    if not mc.empty:
        idata_mc = build_monte_carlo_inference_data(mc, df_all, n_simulations=25_000)
        print(f"✓ MC InferenceData: {summarize_inference_data(idata_mc)}")
    if not beat.empty and "posterior_alpha" in beat.columns:
        idata_beat = build_beat_probability_inference_data(beat, df_all, n_posterior_samples=4000, n_chains=4)
        print(f"✓ Beat InferenceData: {summarize_inference_data(idata_beat)}")
    if not credit.empty:
        idata_credit = build_credit_risk_inference_data(credit, df_all)
        print(f"✓ Credit Risk InferenceData: {summarize_inference_data(idata_credit)}")
else:
    print("⏭️ ArviZ not available — skipping InferenceData")

## 9. Quality & Risk Visualizations

Deep-dive quality and risk dashboards from `finance_ml.analytics.visualizations.quality_risk`.
Now consuming InferenceData built in Step 8.

In [ ]:
##%% Visualization: Quality-Risk Quadrant
fig = create_quality_risk_quadrant(df)
fig.show()

In [ ]:
##%% Visualization: Distress Early Warning Dashboard
fig = create_distress_early_warning_dashboard(df)
fig.show()

In [ ]:
##%% Visualization: Piotroski F-Score Breakdown
if "piotroski_f_score" in df.columns:
    fig = create_piotroski_fscore_breakdown(df)
    fig.show()

In [ ]:
##%% Visualization: Altman Z-Score Distribution
if "altman_z_score" in df.columns:
    fig = create_altman_zscore_distribution(df)
    fig.show()

In [ ]:
##%% Visualization: Beneish M-Score Analysis
fig = create_beneish_mscore_analysis(df)
fig.show()

In [ ]:
##%% Visualization: Risk Tier Sunburst
if "distress_risk_score" in df.columns:
    fig = create_risk_tier_sunburst(df)
    fig.show()

## 10. Export Results

Export all analytics to database, CSV, and JSON in `outputs/analytics/`.
Deduplicated: credit_risk and dividend_safety are exported only once via
`export_expected_returns_results`, not again via `export_probability_analytics_results`.

In [ ]:
##%% Export Functions
from expected_returns_v3 import export_expected_returns_results

In [ ]:
##%% Execute: Export
exports = export_expected_returns_results(
    mc=mc, pt=pt, kal=kal, tri=tri, strong=strong, beat=beat,
    summary=summary, credit=credit, div_safety=div_safety,
    anomaly_results=anomaly_results,
    screens=screens, output_dir=str(output_dir),
)
for name, dest in exports.items():
    print(f"  ✓ {name} → {dest}")

## ✅ Pipeline Summary

In [ ]:
##%% Pipeline Summary
print("=" * 80)
print("✅ EXPECTED RETURNS ANALYTICS v3.1 COMPLETE")
print("=" * 80)
print(f"  mv_equities:              {len(df):,} stocks × {len(df.columns)} features")
print(f"  Monte Carlo:              {len(mc):,}")
print(f"  Price Target Achievement: {len(pt):,}")
print(f"  Kalman Filtered:          {len(kal):,}")
print(f"  Earnings Beat:            {len(beat):,}")
print(f"  Accounting Anomaly:       {len(anomaly_results):,}")
print(f"  Credit Risk:              {len(credit):,}")
print(f"  Tri-model aligned:        {len(tri):,}")
print(f"  Strong consensus:         {len(strong):,}")
print(f"  Summary:                  {len(summary):,}")
print(f"  Categories analyzed:      {len(category_analytics)}")
